# Notebook 04 - Feature Engineering

Goal:
Transform labeled domains into machine-learning features for model training.

Input:
- auspex_domains_v1.csv

Output:
- auspex_features_v1.csv

In [2]:
import pandas as pd

In [3]:
domains = pd.read_csv(
    "../data/processed/auspex_domains_v1.csv"
)

print(domains.shape)

(1024538, 2)


In [5]:
domains.head()

,domain,label
0,cypress.com,benign
1,boy.jp,benign
2,nnm.ru,benign
3,1stwebdesigner.com,benign
4,ordasoft.com,benign


In [6]:
domains["label"].value_counts()

label
benign      988299
malware      26703
phishing      8705
spam           831
Name: count, dtype: int64

In [7]:
domains["length"] = domains["domain"].str.len()

In [8]:
domains[["domain", "length"]].head(20)

,domain,length
0,cypress.com,11
1,boy.jp,6
2,nnm.ru,6
3,1stwebdesigner.com,18
4,ordasoft.com,12
5,stockholm.se,12
6,irena.org,9
7,camcom.it,9
8,timeweb.ru,10
9,waseda.ac.jp,12


In [9]:
domains["length"].describe()

count    1.024538e+06
mean     1.461948e+01
std      5.483131e+00
min      4.000000e+00
25%      1.100000e+01
50%      1.400000e+01
75%      1.700000e+01
max      2.100000e+02
Name: length, dtype: float64

In [10]:
domains.sort_values(
    "length",
    ascending=False
)[["domain", "label", "length"]].head(20)

,domain,label,length
1001748,paypal.de-signin-sicherheit-1544.paypal.de-sig...,malware,210
1001751,paypal.de-signin-sicherheit-7295.paypal.de-sig...,malware,210
1013177,web.bank.of.america.my.wlogin.ab6aacgf40007ddd...,malware,197
1005367,wellsfargousacustomerservice.report.com.ticket...,malware,188
993568,https.www.appleid.com.redirect.redirect.com-my...,phishing,182
995722,www.paypal.com.fr.cgi.bin.webscr.cmd.flow.sess...,phishing,179
1007462,update.com.webscrlcmdl.login.submit.dispatch.5...,malware,178
1001749,paypal.de-signin-sicherheit-2070.amazon.de-sig...,malware,177
1001750,paypal.de-signin-sicherheit-3339.amazon.de-sig...,malware,177
1003592,usaa.com.inet.entlogon.logon.redirectjsp.ef4bc...,malware,176


In [11]:
domains.groupby("label")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
benign,988299.0,14.388455,4.985146,4.0,11.0,14.0,17.0,68.0
malware,26703.0,21.182302,12.000881,5.0,14.0,18.0,24.0,210.0
phishing,8705.0,20.533716,9.987609,4.0,15.0,19.0,24.0,182.0
spam,831.0,16.529483,4.562950,7.0,13.0,16.0,19.0,36.0


### Domain Length

Average domain length: ~14.6 characters.

The longest observed domains were overwhelmingly malware and phishing domains,
often containing brand impersonation strings and excessive subdomain nesting.

Domain length appears to be a potentially useful predictive feature.

### Feature Observation: Domain Length

Domain length differs significantly across classes.

Average lengths:

- benign: 14.4
- spam: 16.5
- malware: 21.2
- phishing: 20.5

Malware and phishing domains are substantially longer than benign domains, suggesting domain length is a useful predictive feature.

In [12]:
domains["digit_count"] = domains["domain"].str.count(r"\d")

In [13]:
domains[["domain", "digit_count"]].sample(10)

,domain,digit_count
21914,donland.ru,0
147510,openinventionnetwork.com,0
468019,31hzp.com,2
667084,infolive.tv,0
561591,leprous.net,0
792360,townsquared.com,0
89988,unecatef.fr,0
150409,freewordpressthemes4u.com,1
872638,11.me,2
449128,clarkdietrich.com,0


In [14]:
domains["digit_ratio"] = (
    domains["digit_count"] / domains["length"]
)

In [15]:
domains["dot_count"] = domains["domain"].str.count(r"\.")

In [16]:
domains["tld"] = domains["domain"].str.split(".").str[-1]

In [17]:
domains["tld"].value_counts().head(20)

tld
com      513136
org       69274
net       53605
ru        36820
cn        36413
uk        24379
de        22311
jp        14565
world     11943
nl        10916
pl        10861
info      10678
it         9556
fr         9020
br         8460
au         7948
us         7309
in         6492
ca         6207
es         5115
Name: count, dtype: int64

In [18]:
domains["starts_with_digit"] = (
    domains["domain"]
    .str[0]
    .str.isdigit()
    .astype(int)
)

In [19]:
domains["contains_www"] = (
    domains["domain"]
    .str.contains("www", case=False)
    .astype(int)
)

In [23]:
domains["hyphen_count"] = domains["domain"].str.count("-")

In [24]:
import math
from collections import Counter

def shannon_entropy(text):
    counts = Counter(text)
    probs = [c / len(text) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

domains["entropy"] = domains["domain"].apply(
    shannon_entropy
)

In [25]:
domains.head()

,domain,label,length,digit_count,digit_ratio,dot_count,tld,starts_with_digit,contains_www,entropy,hyphen_count
0,cypress.com,benign,11,0,0.000000,1,com,0,0,3.095795,0
1,boy.jp,benign,6,0,0.000000,1,jp,0,0,2.584963,0
2,nnm.ru,benign,6,0,0.000000,1,ru,0,0,2.251629,0
3,1stwebdesigner.com,benign,18,1,0.055556,1,com,1,0,3.794653,0
4,ordasoft.com,benign,12,0,0.000000,1,com,0,0,3.188722,0


In [26]:
domains.info()

<class 'pandas.DataFrame'>
RangeIndex: 1024538 entries, 0 to 1024537
Data columns (total 11 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   domain             1024538 non-null  str    
 1   label              1024538 non-null  str    
 2   length             1024538 non-null  int64  
 3   digit_count        1024538 non-null  int64  
 4   digit_ratio        1024538 non-null  float64
 5   dot_count          1024538 non-null  int64  
 6   tld                1024538 non-null  object 
 7   starts_with_digit  1024538 non-null  int64  
 8   contains_www       1024538 non-null  int64  
 9   entropy            1024538 non-null  float64
 10  hyphen_count       1024538 non-null  int64  
dtypes: float64(2), int64(6), object(1), str(2)
memory usage: 86.0+ MB


In [27]:
domains.to_csv(
    "../data/processed/auspex_features_v1.csv",
    index=False
)

# Notebook 04 Summary

Generated engineered features:

- length
- digit_count
- digit_ratio
- hyphen_count
- dot_count
- tld
- starts_with_digit
- contains_www
- entropy

Output:
auspex_features_v1.csv